# 8.14 — Efficient and long-context attention

Long-context attention is the art of keeping the links that matter while refusing to store a full square table just because the formula permits it.

Dense self-attention creates T squared scores before values are mixed. Sparse masks, global routes, deterministic random links, and streaming softmax keep long documents possible on CPU-sized examples.

Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 713
random.seed(SEED)
np.random.seed(SEED)


def softmax(values):
    arr = np.array(values, dtype=float)
    shifted = arr - np.max(arr)
    exp = np.exp(shifted)
    return exp / exp.sum()


def accuracy(pred, gold):
    return sum(int(a == b) for a, b in zip(pred, gold)) / max(1, len(gold))


def show_table(rows, headers):
    widths = [len(str(h)) for h in headers]
    for row in rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(str(value)))
    print(" | ".join(str(h).ljust(widths[i]) for i, h in enumerate(headers)))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(str(value).ljust(widths[i]) for i, value in enumerate(row)))


def make_sequence_ladder(topic):
    if topic == "8.13":
        return [
            {"name": "D1 reversal", "sequences": [["dog", "bites"], ["bites", "dog"]], "labels": [1, 0], "max_pos": 2},
            {"name": "D2 clean order", "sequences": [["a", "then", "b"], ["b", "then", "a"], ["red", "before", "blue"], ["blue", "before", "red"], ["left", "right"]], "labels": [1, 0, 1, 0, 1], "max_pos": 5},
            {"name": "D3 agreement", "sequences": [["dogs", "chase", "cat"], ["cat", "chase", "dogs"], ["he", "likes", "tea"], ["tea", "likes", "he"], ["birds", "fly", "today"]], "labels": [1, 0, 1, 0, 1], "max_pos": 8},
            {"name": "D4 pair tasks", "sequences": [["subject", "verb", "object", "now"], ["object", "verb", "subject", "now"], ["query", "key", "value", "end"], ["value", "key", "query", "end"], ["first", "middle", "last", "stop"]], "labels": [1, 0, 1, 0, 1], "max_pos": 10},
            {"name": "D5 beyond learned", "sequences": [["start"] + ["filler"] * n + ["end"] for n in [4, 8, 12, 16, 20]], "labels": [1, 1, 1, 1, 1], "max_pos": 22},
        ]
    if topic == "8.14":
        return [
            {"name": "D1 mask toy", "T": 32, "w": 2, "evidence": [(19, 0)], "global_tokens": []},
            {"name": "D2 local deps", "T": 24, "w": 2, "evidence": [(4, 3), (8, 6), (12, 11), (16, 15), (20, 18)], "global_tokens": []},
            {"name": "D3 global random", "T": 40, "w": 2, "evidence": [(30, 0), (35, 1), (21, 18), (10, 8), (39, 3)], "global_tokens": [0, 1]},
            {"name": "D4 document snippets", "T": 64, "w": 3, "evidence": [(50, 2), (60, 4), (32, 31), (12, 10), (45, 5)], "global_tokens": [0, 2, 4]},
            {"name": "D5 far evidence", "T": 96, "w": 3, "evidence": [(90, 0), (88, 6), (70, 9), (44, 43), (30, 27)], "global_tokens": [0, 6, 9]},
        ]
    if topic == "8.15":
        return [
            {"name": "D1 logits", "rows": [[3.0, 2.0, 0.5]], "target": ["A"]},
            {"name": "D2 clean rows", "rows": [[3, 1, 0], [0, 3, 1], [1, 0, 3], [4, 2, 0], [0, 4, 2]], "target": list("ABCAB")},
            {"name": "D3 traps", "rows": [[2.1, 2.0, 0], [1.8, 1.7, 1.6], [3, 2.9, 0], [2, 2, 1.9], [0, 2.2, 2.1]], "target": list("BACCB")},
            {"name": "D4 candidates", "rows": [[2.5, 2.4, 0.1], [0.5, 2.0, 1.9], [2.2, 2.1, 2.0], [3.0, 0.2, 0.1], [0.1, 2.8, 2.7]], "target": list("ABABB")},
            {"name": "D5 long generation", "rows": [[2.0, 1.9, 0.2], [1.5, 1.4, 2.2], [2.1, 2.0, 0.4], [0.4, 2.2, 2.1], [1.9, 1.8, 0.7], [0.5, 2.4, 2.3], [2.3, 2.2, 0.2]], "target": list("ACABABB")},
        ]
    if topic == "8.16":
        return [
            {"name": "D1 lesson pair", "cand": "the cat sat", "ref": "the cat slept", "probs": [0.5, 0.25, 0.25]},
            {"name": "D2 paraphrases", "pairs": [("small cat sleeps", "small cat sleeps"), ("red bird flies", "red bird sings"), ("quick dog runs", "dog runs quickly"), ("bright moon rises", "moon rises bright"), ("blue car stops", "blue bus stops")]},
            {"name": "D3 predicate order", "pairs": [("the cat chased mouse", "the mouse chased cat"), ("doctor treats patient", "doctor helps patient"), ("team wins final", "team loses final"), ("user resets password", "password resets user"), ("model answers question", "model answers query")]},
            {"name": "D4 summaries", "pairs": [("sales rose in june", "sales rose in june after ads"), ("server failed at noon", "server recovered at noon"), ("policy changed for users", "policy changed for all users"), ("campaign reached creators", "campaign reached many creators"), ("video views increased", "video watch time increased")]},
            {"name": "D5 long refs", "pairs": [("the model answered safely with citations", "the model answered correctly with citations"), ("the summary covered three facts and omitted risk", "the summary covered three facts and named risk"), ("translation kept names but changed tense", "translation kept names and preserved tense"), ("caption mentions dog running near beach", "caption mentions dog walking near beach"), ("assistant refused unsafe request politely", "assistant refused unsafe request clearly")]},
        ]
    if topic == "8.17":
        return [
            {"name": "D1 Ada", "tokens": ["Ada", "works", "at", "OpenAI"], "tags": ["B-PER", "O", "O", "B-ORG"]},
            {"name": "D2 clean entities", "examples": [(["Ada", "joined", "OpenAI"], ["B-PER", "O", "B-ORG"]), (["Lin", "visits", "Paris"], ["B-PER", "O", "B-LOC"]), (["DeepMind", "hired", "Ilya"], ["B-ORG", "O", "B-PER"]), (["Mira", "met", "Sam"], ["B-PER", "O", "B-PER"]), (["Google", "opened", "office"], ["B-ORG", "O", "O"])]},
            {"name": "D3 adjacent padding", "examples": [(["Ada", "Bob", "spoke"], ["B-PER", "B-PER", "O"]), (["New", "York", "Labs"], ["B-LOC", "I-LOC", "B-ORG"]), (["Eve", "from", "IBM", "arrived"], ["B-PER", "O", "B-ORG", "O"]), (["Paris", "Paris", "team"], ["B-LOC", "B-ORG", "O"]), (["Zed", "Corp", "LLC"], ["B-ORG", "I-ORG", "I-ORG"])]},
            {"name": "D4 snippets", "examples": [(["Dr", "Chen", "at", "Mayo"], ["B-PER", "I-PER", "O", "B-ORG"]), (["Acme", "bought", "Beta"], ["B-ORG", "O", "B-ORG"]), (["Nina", "left", "Berlin"], ["B-PER", "O", "B-LOC"]), (["OpenAI", "San", "Francisco"], ["B-ORG", "B-LOC", "I-LOC"]), (["Raj", "from", "UN"], ["B-PER", "O", "B-ORG"])]},
            {"name": "D5 long multi", "examples": [(["Ada", "Lovelace", "worked", "with", "Babbage", "Labs"], ["B-PER", "I-PER", "O", "O", "B-ORG", "I-ORG"]), (["Maya", "from", "OpenAI", "visited", "London"], ["B-PER", "O", "B-ORG", "O", "B-LOC"]), (["IBM", "and", "Google", "met", "in", "Paris"], ["B-ORG", "O", "B-ORG", "O", "O", "B-LOC"]), (["Dr", "Lee", "joined", "Mayo", "Clinic"], ["B-PER", "I-PER", "O", "B-ORG", "I-ORG"]), (["Sam", "Altman", "visited", "New", "York"], ["B-PER", "I-PER", "O", "B-LOC", "I-LOC"])]},
        ]
    return [
        {"name": "D1 morphology", "tokens": ["walk", "walked", "walking"], "tags": ["VERB", "VERB", "VERB"]},
        {"name": "D2 clean POS", "examples": [(["dogs", "walk"], ["NOUN", "VERB"]), (["they", "walked"], ["NOUN", "VERB"]), (["walking", "helps"], ["NOUN", "VERB"]), (["red", "dogs", "run"], ["ADJ", "NOUN", "VERB"]), (["cats", "sleep"], ["NOUN", "VERB"])]},
        {"name": "D3 ambiguous", "examples": [(["walk", "home"], ["VERB", "NOUN"]), (["the", "walk"], ["DET", "NOUN"]), (["walking", "tour"], ["ADJ", "NOUN"]), (["we", "duck"], ["NOUN", "VERB"]), (["duck", "soup"], ["NOUN", "NOUN"])]},
        {"name": "D4 tagged lines", "examples": [(["bright", "birds", "sing"], ["ADJ", "NOUN", "VERB"]), (["users", "liked", "ads"], ["NOUN", "VERB", "NOUN"]), (["fast", "models", "serve"], ["ADJ", "NOUN", "VERB"]), (["the", "running", "joke"], ["DET", "ADJ", "NOUN"]), (["creators", "earned", "money"], ["NOUN", "VERB", "NOUN"])]},
        {"name": "D5 agreement", "examples": [(["the", "dogs", "were", "walking"], ["DET", "NOUN", "VERB", "VERB"]), (["a", "walk", "was", "short"], ["DET", "NOUN", "VERB", "ADJ"]), (["walking", "brands", "sell"], ["ADJ", "NOUN", "VERB"]), (["names", "ending", "ing", "confuse"], ["NOUN", "VERB", "NOUN", "VERB"]), (["past", "users", "walked"], ["ADJ", "NOUN", "VERB"])]},
    ]



def attention_mask(T, w, globals=None, random_links=None):
    globals = set(globals or [])
    random_links = set(random_links or [])
    mask = np.zeros((T, T), dtype=bool)
    for i in range(T):
        for j in range(T):
            if abs(i - j) <= w:
                mask[i, j] = True
            if i in globals or j in globals:
                mask[i, j] = True
            if (i, j) in random_links:
                mask[i, j] = True
    return mask


def deterministic_random_links(T, count, seed=SEED):
    rng = random.Random(seed + T + count)
    links = set()
    while len(links) < count:
        i = rng.randrange(T)
        j = rng.randrange(T)
        if abs(i - j) > 3:
            links.add((i, j))
    return links


def retrieval_accuracy(rung, use_globals=True, use_random=True):
    random_links = deterministic_random_links(rung["T"], 12) if use_random else set()
    globals_used = rung["global_tokens"] if use_globals else []
    mask = attention_mask(rung["T"], rung["w"], globals_used, random_links)
    hits = [bool(mask[q, k]) for q, k in rung["evidence"]]
    return sum(hits) / len(hits), mask


def streaming_softmax(scores, block=2):
    m = -float("inf")
    denom = 0.0
    chunks = [scores[i:i + block] for i in range(0, len(scores), block)]
    for chunk in chunks:
        chunk = np.array(chunk, dtype=float)
        new_m = max(m, float(np.max(chunk)))
        denom = denom * math.exp(m - new_m) + float(np.exp(chunk - new_m).sum())
        m = new_m
    return np.exp(np.array(scores) - m) / denom


## The concept, built once: mask the score matrix

Dense attention stores $T^2$ scores. A window mask is $M_{ij}=\mathbf{1}\{|i-j|\le w\}$, with boundary effects that make $T=32,w=2$ equal 154 entries rather than the approximation 160.

In [ ]:

lengths = [128, 256, 512, 1024]
squares = [t * t for t in lengths]
mask = attention_mask(32, 2)
allowed = int(mask.sum())
print(squares)
print("allowed", allowed)
assert squares == [16384, 65536, 262144, 1048576]
assert allowed == 154


Global tokens and deterministic random links add explicit long routes. FlashAttention is different: it keeps dense attention exact by streaming the global softmax denominator rather than keeping only local chunk maxima.

In [ ]:

target_probs = [0.665186, 0.244708, 0.090023, 0.000082]
scores = [math.log(p) for p in target_probs]
probs = streaming_softmax(scores, block=2)
local_only = np.concatenate([softmax(scores[:2]), softmax(scores[2:])]) / 2
print(np.round(probs, 6))
print(np.round(local_only, 6))
assert np.round(probs, 6).tolist() == target_probs
assert bool(attention_mask(20, 2)[19, 0]) is False


## The dataset ladder

D1 is the lesson mask, then local dependencies, global/random routes, document snippets, and D5 far evidence.

In [ ]:

ladder = make_sequence_ladder("8.14")
for rung in ladder:
    print(rung["name"], "T=", rung["T"], "w=", rung["w"], "globals=", rung["global_tokens"], "first evidence=", rung["evidence"][0])


In [ ]:

rows = []
metrics = []
edges = []
for rung in ladder:
    score, mask = retrieval_accuracy(rung)
    metrics.append(score)
    edges.append(int(mask.sum()))
    rows.append([rung["name"], rung["T"], int(mask.sum()), round(score, 3)])
show_table(rows, ["rung", "T", "edges", "accuracy"])


In [ ]:

fig, axes = plt.subplots(1, 6, figsize=(15, 2.4))
for ax, rung in zip(axes[:5], ladder):
    _, mask = retrieval_accuracy(rung)
    ax.imshow(mask[:40, :40], cmap="Greys", aspect="auto")
    ax.set_title(rung["name"].split()[0])
    ax.set_xlabel("key")
    ax.set_ylabel("query")
axes[5].plot(edges, metrics, marker="o")
axes[5].set_ylim(0, 1.05)
axes[5].set_title("accuracy per edge budget")
axes[5].set_xlabel("allowed edges")
fig.tight_layout()


## Pitfall on D5: sparse is not dense

Local windows alone remove far evidence. The fix is an explicit global route plus seeded random links so reachability is reproducible.

In [ ]:

d5 = ladder[-1]
local_score, local_mask = retrieval_accuracy(d5, use_globals=False, use_random=False)
fixed_score, fixed_mask = retrieval_accuracy(d5, use_globals=True, use_random=True)
print("local-only accuracy", local_score)
print("fixed accuracy", fixed_score)
print("M[90,0] local", bool(local_mask[90, 0]))
print("M[90,0] fixed", bool(fixed_mask[90, 0]))
assert local_score < fixed_score
assert bool(local_mask[90, 0]) is False
assert bool(fixed_mask[90, 0]) is True


## Evaluate it + Practice

- Metric: evidence-retrieval accuracy per allowed edge budget; no-skill baseline is local-only reachability.
- Sanity check: $T=32,w=2$ gives exactly 154 mask entries.
- Ablation: remove global routes and far evidence should fail.
- Failure signal: streamed softmax differs from dense softmax when chunk denominators are not merged.

Practice: add two global tokens to D3 and count the new edges.

Practice: change the random seed and inspect which evidence pairs become reachable.